In [1]:
import os
# ============================================================================
# 1. Rutas (cámbialas)
# ============================================================================
ruta_precip = '/home/santiago/Escritorio/UNIVERSIDAD/TRABAJO_DE_GRADO/DATOS/Precipitacion_SA.nc'
ruta_nino = '/home/santiago/Escritorio/UNIVERSIDAD/TRABAJO_DE_GRADO/DATOS/datos_nino.xlsx'
carpeta_salida = '/home/santiago/Escritorio/UNIVERSIDAD/TRABAJO_DE_GRADO/DATOS' 
os.makedirs(carpeta_salida, exist_ok=True)

In [2]:
import xarray as xr
import pandas as pd
import numpy as np
import os
import psutil
import gc
from tqdm import tqdm

# ============================================================================
# FUNCIÓN PARA MONITOREAR MEMORIA
# ============================================================================
def print_memory_usage(stage):
    mem = psutil.Process().memory_info().rss / 1024 / 1024
    print(f"[MEMORIA] {stage}: {mem:.1f} MB")



# ============================================================================
# 2. CARGAR PRECIPITACIÓN Y REDUCIR RESOLUCIÓN ESPACIAL
# ============================================================================
print("Cargando precipitación...")
ds = xr.open_dataset(ruta_precip)
ds = ds.rename({'valid_time': 'time', 'latitude': 'lat', 'longitude': 'lon'})
precip = ds['tp'] * 1000   # mm/mes
print_memory_usage("Después de cargar precipitación original")

# --- REDUCCIÓN ESPACIAL (AJUSTA EL FACTOR SEGÚN TU MEMORIA) ---
coarsen_factor = 2   # 2 = 140×148, 3 = ~93×99, 4 = 70×74
precip_coarse = precip.coarsen(lat=coarsen_factor, lon=coarsen_factor, boundary='trim').mean()
print(f"Nueva dimensión espacial: {precip_coarse.lat.size} lat × {precip_coarse.lon.size} lon")
print_memory_usage("Después de coarsen")

# Recortar hasta marzo de 2025 (opcional)
# Primero normalizamos las fechas a inicio de mes para que coincidan con ENSO
precip_coarse['time'] = precip_coarse.time.dt.floor('D')  # elimina la parte horaria (06:00:00)
precip_coarse = precip_coarse.sel(time=slice('1950-01-01', '2025-03-01'))
print(f"Rango temporal: {precip_coarse.time.min().values} → {precip_coarse.time.max().values}")
print_memory_usage("Después de recorte temporal")

# ============================================================================
# 3. CLIMATOLOGÍA MENSUAL (1991-2020)
# ============================================================================
print("Calculando climatología...")
clim_start, clim_end = 1991, 2020
mask_clim = (precip_coarse.time.dt.year >= clim_start) & (precip_coarse.time.dt.year <= clim_end)

clim_mean = precip_coarse.where(mask_clim).groupby('time.month').mean('time')
clim_std  = precip_coarse.where(mask_clim).groupby('time.month').std('time')
print_memory_usage("Después de climatología")

# ============================================================================
# 4. CARGAR ÍNDICE ENSO 3.4
# ============================================================================
print("Cargando ENSO...")
df = pd.read_excel(ruta_nino)

# Verificar columnas
if 'YR' not in df.columns or 'MON' not in df.columns:
    raise ValueError("El archivo Excel debe tener columnas 'YR' y 'MON'")

# Crear fechas correctamente
df_fechas = df[['YR', 'MON']].rename(columns={'YR': 'year', 'MON': 'month'})
df_fechas['day'] = 1
df['date'] = pd.to_datetime(df_fechas)

# Seleccionar la columna (ANOM3.4 o NINO3.4)
enso = df.set_index('date')['ANOM3.4']   # o 'NINO3.4'

# Renombrar índice para evitar conflicto
enso.index = enso.index.rename('time')

# Convertir a xarray DataArray
enso_da = xr.DataArray(enso, dims=['time'])
print_memory_usage("Después de cargar ENSO")

# ============================================================================
# 5. ALINEAR PERÍODOS
# ============================================================================
# Asegurar que ambas series tengan el mismo formato de tiempo (inicio de mes)
# Precipitación ya tiene floor('D')
# ENSO tiene día 1, así que ambas coinciden

# Recortar al período común
time_start = max(precip_coarse.time.min().values, enso_da.time.min().values)
time_end = min(precip_coarse.time.max().values, enso_da.time.max().values)

precip_sub = precip_coarse.sel(time=slice(time_start, time_end))
enso_sub = enso_da.sel(time=slice(time_start, time_end))

print(f"Período común: {time_start} a {time_end}")
print_memory_usage("Después de alinear ENSO y precipitación")

# ============================================================================
# 6. CORRELACIÓN CELDA POR CELDA
# ============================================================================
print("Calculando correlación celda a celda...")
latitudes = precip_sub.lat.values
longitudes = precip_sub.lon.values
corr_matrix = np.full((len(latitudes), len(longitudes)), np.nan, dtype=np.float32)

# Obtener la serie ENSO como numpy
enso_values = enso_sub.values

for i, lat in enumerate(tqdm(latitudes, desc="Latitudes")):
    for j, lon in enumerate(longitudes):
        prec_series = precip_sub.isel(lat=i, lon=j).values
        mask = ~np.isnan(prec_series)
        if np.sum(mask) > 10:
            corr_matrix[i, j] = np.corrcoef(prec_series[mask], enso_values[mask])[0, 1]
    if i % 20 == 0:
        gc.collect()

print_memory_usage("Después de correlación")

# ============================================================================
# 7. GUARDAR MAPA DE CORRELACIÓN COMO GEOTIFF
# ============================================================================
print("Guardando resultado...")
corr_da = xr.DataArray(
    corr_matrix,
    dims=('lat', 'lon'),
    coords={'lat': latitudes, 'lon': longitudes},
    name='correlacion_enso'
)

# Establecer dimensiones espaciales para rioxarray
corr_da = corr_da.rio.set_spatial_dims(x_dim='lon', y_dim='lat')
corr_da = corr_da.rio.write_crs('EPSG:4326', inplace=True)

ruta_corr = os.path.join(carpeta_salida, f'correlacion_enso_precip_coarse_{coarsen_factor}.tif')
corr_da.rio.to_raster(ruta_corr, compress='DEFLATE', dtype='float32')
print(f"Correlación guardada en: {ruta_corr}")
print_memory_usage("Final")

Cargando precipitación...
[MEMORIA] Después de cargar precipitación original: 909.9 MB
Nueva dimensión espacial: 149 lat × 148 lon
[MEMORIA] Después de coarsen: 970.2 MB
Rango temporal: 1950-01-01T00:00:00.000000000 → 2025-03-01T00:00:00.000000000
[MEMORIA] Después de recorte temporal: 973.2 MB
Calculando climatología...
[MEMORIA] Después de climatología: 976.3 MB
Cargando ENSO...
[MEMORIA] Después de cargar ENSO: 983.3 MB
Período común: 1950-01-01T00:00:00.000000000 a 2025-03-01T00:00:00.000000000
[MEMORIA] Después de alinear ENSO y precipitación: 983.3 MB
Calculando correlación celda a celda...


Latitudes: 100%|██████████| 149/149 [00:05<00:00, 25.15it/s]


[MEMORIA] Después de correlación: 984.9 MB
Guardando resultado...
Correlación guardada en: /home/santiago/Escritorio/UNIVERSIDAD/TRABAJO_DE_GRADO/DATOS/correlacion_enso_precip_coarse_2.tif
[MEMORIA] Final: 1000.1 MB
